In [1]:
import os
from dotenv import load_dotenv
import xarray as xr

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datetime import date

import analysis_utils
import isku_utils

import importlib

importlib.reload(analysis_utils)
importlib.reload(isku_utils)

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<module 'isku_utils' from '/home/emily_zuetell/projects/poreallas/analysis/isku_utils.py'>

In [2]:
load_dotenv()
#DATA_DIR = os.environ["DATA_DIR"]
# Baseline period for Impact
BASELINE_PERIOD = slice("1996-01-01", "2025-12-31")
# Define Forecast Months
FC_MONTHS = [9, 10, 11, 12, 1, 2]
FC_PERIOD = slice("2026-09-01", "2027-02-28")

config = analysis_utils.ImpactConfig( version = "v260910",
                                     baseline_period=BASELINE_PERIOD, 
                                     polygons_path= os.environ["POREALLAS_REGIONS_POLYGONS_URI"],
                                     socioeconomics_path= os.environ["POREALLAS_SOCIOECONOMICS_URI"],
                                     rate=False, 
                                     months = FC_MONTHS,
                                     hotonly = "hotonly", 
                                     dims = ['number', 'sample'])

# Define Forecast
EFFECTS_URI = "/home/emily_zuetell/projects/poreallas/data/v20260909_effects_with_betas.zarr" # Can be any effects datatree

In [3]:
# Projection Effects
effect = xr.open_datatree(EFFECTS_URI, consolidated=False)

In [4]:
### Log baseline period and impact calculation
rate_l = "rate" if config.rate else "total"
baseline_tag = analysis_utils._baseline_tag(config.baseline_period)

In [5]:
# Compute impact: forecast - baseline
impact = analysis_utils.compute_impact(
    effect.chunk({dim: -1 for dim in config.dims}),
    config,
    ensemble=True,
)
# Use only the defined 6-months
impact = impact.sel(month=config.months)

In [12]:
_polygons_impact = analysis_utils.xarray_to_gpd(
    impact.mean(dim = config.dims).sum(dim = 'month'), config.polygons
)

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)


In [14]:
gdf = _polygons_impact
states = gpd.read_file("https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_state_20m.zip")
california = states[states["NAME"] == "California"].to_crs(gdf.crs)

gdf_ca = gpd.clip(gdf, california)

In [16]:
gdf_ca.sort_values("age_weighted_impact", ascending=False)

,region,gadmid,color,ISO,AREA,PERIMETER,geometry,year,age_weighted_impact
13627,USA.5.220,205823,13628,USA,1.063390,4.606778,"POLYGON ((-10416453.309 3537567.605, -10418547...",2026,235.484983
13726,USA.5.202,205922,13727,USA,1.033183,6.136823,"MULTIPOLYGON (((-10498640.74 3696196.495, -105...",2026,206.005725
13585,USA.5.213,205781,13586,USA,0.201357,1.955559,"POLYGON ((-10523451.166 3611581.158, -10519856...",2026,121.063237
13014,USA.5.184,205210,13015,USA,0.200478,2.412653,"POLYGON ((-10709252.849 4014646.471, -10705307...",2026,77.183186
13040,USA.5.224,205236,13041,USA,0.119370,1.864001,"POLYGON ((-10770261.288 4002888.048, -10774549...",2026,58.592378
...,...,...,...,...,...,...,...,...,...
13170,USA.5.237,205366,13171,USA,1.255928,4.926535,"POLYGON ((-10466636.89 3922038.612, -10464339....",2026,-0.767616
12907,USA.5.188,205103,12908,USA,0.275369,2.559836,"POLYGON ((-10544472.68 4104611.501, -10560114....",2026,-0.772339
12103,USA.29.1765,204299,12104,USA,1.803176,7.242793,"MULTIPOLYGON (((-10492181.376 4187618.137, -10...",2026,-1.326300
13342,USA.5.219,205538,13343,USA,5.132250,10.092953,"POLYGON ((-10258034.624 3818145.947, -10252257...",2026,-6.709643


In [6]:
# Aggregate Impact Regions to group_level
group_level = 'ISO' #IR: Impact region, # ADM1: State level, # ISO: Country level
# Use a subset of ensemble members for quick testing (.sel(number = ...))
#impact = impact.sel(number = [0, 1, 2, 3, 4])
impact, merge_key, base_cols = analysis_utils.aggregate_impact(impact, config, group_level)

In [52]:
#Compute stats in xarray from dims in config.dims
stat_cols = ["median", "p17", "p83", "likely_range_IPCC", "mean", "std", "min", "max", "p10", "p90"]
_polygons_impact = analysis_utils.dataset_to_dataframe(analysis_utils.compute_stats(impact, dim=config.dims))

In [54]:
filename_template="{version}_{hotonly}_{scope}_{rate_l}_{stat_scope}_{group_level}_{baseline}_{cleaned}.csv"
# "regional_monthly" 
_polygons_impact = analysis_utils.dataset_to_dataframe(analysis_utils.compute_stats(impact, dim=config.dims))
wide = _polygons_impact.pivot(
    index=base_cols,
    columns="month", values=stat_cols,
)
wide.columns = [f"month {m} {stat}" for stat, m in wide.columns]
stat_col_names = wide.columns.difference(base_cols)
wide = wide.reset_index()
wide_rounded = analysis_utils.round_output(wide)
wide.to_csv(
    filename_template.format(
        version=config.version,
        hotonly=config.hotonly,
        rate_l=rate_l,
        scope="monthly",
        stat_scope="",
        group_level=group_level,
        baseline=baseline_tag,
        cleaned = 'raw'
    ),
    index=False,
)
wide_rounded.to_csv(
    filename_template.format(
        version=config.version,
        hotonly=config.hotonly,
        rate_l=rate_l,
        scope="monthly",
        stat_scope="",
        group_level=group_level,
        baseline=baseline_tag,
        cleaned = 'rounded'
    ),
    index=False,
)
